In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torchvision import models
from torch.utils.data import DataLoader, Dataset
from PIL import Image
import json
import time
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

with open("C:/SmartMeal/src/config.json") as f:
    config = json.load(f)

IMAGES_DIR  = Path(config["images_dir"])
CLASSES     = config["classes"]  # 101 classes
DEVICE      = torch.device("cpu")

print(f"✅ {len(CLASSES)} classes chargées")
print(f"🖥️  Device : {DEVICE}")

✅ 101 classes chargées
🖥️  Device : cpu


In [2]:
# ── Optimisé pour entraînement overnight CPU ──────────────
BATCH_SIZE  = 32   # plus grand = plus rapide
NUM_EPOCHS  = 15   # suffisant pour transfer learning
LR          = 0.001
IMG_SIZE    = 224

# Estimation temps :
# 75,750 images / 32 batch = 2,368 batches
# ~3 sec/batch sur CPU = ~7,100 sec/epoch = ~2h/epoch
# 15 epochs = ~30h  ← trop long !

# ── Solution : utiliser seulement 30% des données ─────────
SUBSET_RATIO = 0.3  # 30% = ~22,700 images train
                    # ~2h/epoch × 15 = ~8h overnight ✅

print(f"📊 Stratégie entraînement :")
print(f"   Total train    : 75,750 images")
print(f"   Subset (30%)   : {int(75750 * SUBSET_RATIO):,} images")
print(f"   Batch size     : {BATCH_SIZE}")
print(f"   Epochs         : {NUM_EPOCHS}")
print(f"   Estimé         : ~6-10h (overnight) ✅")

📊 Stratégie entraînement :
   Total train    : 75,750 images
   Subset (30%)   : 22,725 images
   Batch size     : 32
   Epochs         : 15
   Estimé         : ~6-10h (overnight) ✅


In [3]:
class Food101Dataset(Dataset):
    def __init__(self, images_dir, classes, split_file, 
                 transform=None, subset_ratio=1.0):
        self.transform    = transform
        self.samples      = []
        self.class_to_idx = {cls: i for i, cls in enumerate(classes)}

        meta_dir = images_dir.parent / "meta"
        with open(meta_dir / split_file) as f:
            lines = [l.strip() for l in f.readlines()]

        # Subset aléatoire reproductible
        import random
        random.seed(42)
        if subset_ratio < 1.0:
            lines = random.sample(lines, int(len(lines) * subset_ratio))

        for line in lines:
            cls      = line.split("/")[0]
            img_path = images_dir / f"{line}.jpg"
            if img_path.exists() and cls in self.class_to_idx:
                self.samples.append((img_path, self.class_to_idx[cls]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label

# ── Transformations ───────────────────────────────────────
train_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

test_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

print("⏳ Chargement datasets...")
train_dataset = Food101Dataset(IMAGES_DIR, CLASSES, "train.txt",
                               train_transforms, SUBSET_RATIO)
test_dataset  = Food101Dataset(IMAGES_DIR, CLASSES, "test.txt",
                               test_transforms, 0.2)

train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                           shuffle=True,  num_workers=0)
test_loader   = DataLoader(test_dataset,  batch_size=BATCH_SIZE,
                           shuffle=False, num_workers=0)

print(f"✅ Train : {len(train_dataset):,} images")
print(f"✅ Test  : {len(test_dataset):,}  images")
print(f"📦 Batches train : {len(train_loader)}")

⏳ Chargement datasets...
✅ Train : 22,725 images
✅ Test  : 5,050  images
📦 Batches train : 711


In [4]:
print("⬇️  Chargement ResNet-18...")

model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

# Geler toutes les couches sauf les 2 derniers blocs + fc
# → plus de paramètres entraînés = meilleure accuracy
for name, param in model.named_parameters():
    if "layer4" in name or "fc" in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

# Nouvelle tête pour 101 classes
model.fc = nn.Sequential(
    nn.Linear(model.fc.in_features, 512),
    nn.ReLU(),
    nn.Dropout(0.4),
    nn.Linear(512, 101)
)

model = model.to(DEVICE)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"✅ ResNet-18 — 101 classes prêt")
print(f"   Total params     : {total:,}")
print(f"   Trainable params : {trainable:,}  ({100*trainable/total:.1f}%)")
print(f"   Couches libérées : layer4 + fc")

⬇️  Chargement ResNet-18...
✅ ResNet-18 — 101 classes prêt
   Total params     : 11,490,981
   Trainable params : 8,708,197  (75.8%)
   Couches libérées : layer4 + fc


In [5]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR, weight_decay=1e-4
)
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=NUM_EPOCHS
)

history    = {"train_loss": [], "train_acc": [], "test_acc": []}
best_acc   = 0.0
models_dir = Path("C:/SmartMeal/models")

print(f"🌙 Entraînement overnight — {NUM_EPOCHS} epochs")
print(f"   Lance ça avant de dormir !\n")

for epoch in range(NUM_EPOCHS):
    # ── Train ─────────────────────────────────────────────
    model.train()
    running_loss = correct = total = 0
    t0 = time.time()

    for batch_idx, (images, labels) in enumerate(train_loader):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted  = outputs.max(1)
        total        += labels.size(0)
        correct      += predicted.eq(labels).sum().item()

        if (batch_idx + 1) % 100 == 0:
            print(f"  E{epoch+1} | B{batch_idx+1}/{len(train_loader)} "
                  f"| Loss:{running_loss/(batch_idx+1):.3f} "
                  f"| Acc:{100.*correct/total:.1f}%")

    # ── Eval ──────────────────────────────────────────────
    model.eval()
    test_correct = test_total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs        = model(images)
            _, predicted   = outputs.max(1)
            test_total    += labels.size(0)
            test_correct  += predicted.eq(labels).sum().item()

    train_acc  = 100. * correct / total
    test_acc   = 100. * test_correct / test_total
    epoch_loss = running_loss / len(train_loader)
    elapsed    = time.time() - t0

    history["train_loss"].append(epoch_loss)
    history["train_acc"].append(train_acc)
    history["test_acc"].append(test_acc)
    scheduler.step()

    print(f"\n✅ Epoch {epoch+1}/{NUM_EPOCHS} | "
          f"Loss:{epoch_loss:.3f} | "
          f"Train:{train_acc:.1f}% | "
          f"Test:{test_acc:.1f}% | "
          f"⏱️{elapsed/60:.1f}min\n")

    # Sauvegarder le meilleur modèle
    if test_acc > best_acc:
        best_acc = test_acc
        torch.save({
            "model_state_dict" : model.state_dict(),
            "classes"          : CLASSES,
            "history"          : history,
            "best_acc"         : best_acc,
            "epoch"            : epoch
        }, models_dir / "resnet18_food101.pth")
        print(f"   💾 Meilleur modèle sauvegardé ! ({best_acc:.1f}%)")

print(f"\n🎉 Entraînement terminé !")
print(f"   Meilleure accuracy : {best_acc:.1f}%")

🌙 Entraînement overnight — 15 epochs
   Lance ça avant de dormir !

  E1 | B100/711 | Loss:4.263 | Acc:8.7%
  E1 | B200/711 | Loss:4.019 | Acc:12.4%


KeyboardInterrupt: 